# HYDRA-Net Stage 3: Train Swarm-Reasoning GNN

**Notebook purpose:** Train the Stage 3 graph neural network for multi-drone (swarm) reasoning.

**Runtime:** GPU recommended but optional.

**Dataset situation:** Public labeled swarm datasets are scarce. We use a hybrid approach:
1. Generate synthetic swarm scenes programmatically (coordinated formations, dispersed flocks, single-drone scenes)
2. Augment with single-drone embeddings from the trained Stage 2 model, composed into multi-drone scenes
3. Optional: apply MDS or IARPA Perseus swarm datasets if you have access

This is an openly acknowledged research gap — training Stage 3 on truly realistic swarm data will be a contribution in itself.


## 1. Setup


In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())

!git clone https://github.com/YOUR-USERNAME/hydra-net.git
%cd hydra-net
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, 'src')

## 2. Synthetic swarm scene generator

Generate training scenes with diverse topology:
- Single-drone (control)
- Coordinated formation (V-shape, line, box) — high threat
- Dispersed cluster (uncoordinated birds or hobbyists) — low threat
- Attacking formation (converging on a target) — very high threat


In [ ]:
import numpy as np
import torch
from hydra_net.stage3 import build_adjacency_from_kinematics, INTENT_CLASSES, ACTION_CLASSES


def generate_scene(scene_type: str, rng: np.random.Generator):
    """Return (node_feats, positions, velocities, per_drone_threat, intent_labels, action_labels)."""
    if scene_type == 'single':
        n = 1
        positions = rng.uniform(-50, 50, (1, 3))
        velocities = rng.uniform(-5, 5, (1, 3))
        threat = np.array([rng.uniform(1.0, 3.0)])
        intent = np.array([rng.choice([0, 1, 4])])  # benign/surveillance/unknown
        action = np.array([1])  # alert_operator
    elif scene_type == 'formation':
        n = rng.integers(3, 7)
        # Line formation moving together
        base = rng.uniform(-30, 30, (1, 3))
        offsets = np.arange(n).reshape(-1, 1) * np.array([[10, 0, 0]])
        positions = base + offsets + rng.normal(0, 1, (n, 3))
        common_v = rng.uniform(-8, 8, (1, 3))
        velocities = np.tile(common_v, (n, 1)) + rng.normal(0, 0.5, (n, 3))
        threat = np.full(n, rng.uniform(3.5, 5.0))
        intent = np.full(n, 3)  # attack
        action = np.full(n, 4)  # intercept
    elif scene_type == 'dispersed':
        n = rng.integers(3, 8)
        positions = rng.uniform(-60, 60, (n, 3))
        velocities = rng.uniform(-6, 6, (n, 3))  # independent
        threat = rng.uniform(0.5, 2.0, n)
        intent = np.full(n, 0)  # benign
        action = np.full(n, 0)  # monitor
    else:  # attacking: converging
        n = rng.integers(3, 6)
        positions = rng.uniform(-80, 80, (n, 3))
        target = np.array([0., 0., 20.])
        velocities = (target - positions) * rng.uniform(0.05, 0.15, (n, 1))
        threat = np.full(n, 4.8)
        intent = np.full(n, 3)  # attack
        action = np.full(n, 4)  # intercept

    # Node features: [stage2 embedding (128), pos(3), vel(3), per-modality confidence (3)]
    stage2_emb = rng.normal(0, 1, (n, 128))
    confidences = rng.uniform(0.6, 1.0, (n, 3))
    node_feats = np.concatenate([stage2_emb, positions, velocities, confidences], axis=1)

    return (
        torch.tensor(node_feats, dtype=torch.float32),
        torch.tensor(positions, dtype=torch.float32),
        torch.tensor(velocities, dtype=torch.float32),
        torch.tensor(threat, dtype=torch.float32),
        torch.tensor(intent, dtype=torch.long),
        torch.tensor(action, dtype=torch.long),
    )

# Test
rng = np.random.default_rng(0)
for scene_type in ['single', 'formation', 'dispersed', 'attacking']:
    out = generate_scene(scene_type, rng)
    print(f'{scene_type}: n_drones={out[0].size(0)}, mean_threat={out[3].mean():.2f}')

## 3. Train Stage 3 GNN


In [ ]:
from hydra_net.stage3 import SwarmReasoningNetwork
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SwarmReasoningNetwork(node_dim=128+3+3+3, hidden_dim=128, n_layers=3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

N_SCENES = 5000
rng = np.random.default_rng(42)
scene_types = ['single', 'formation', 'dispersed', 'attacking']

model.train()
for step in range(N_SCENES):
    scene_type = rng.choice(scene_types)
    node_feats, positions, velocities, threat, intent, action = generate_scene(scene_type, rng)
    node_feats, threat = node_feats.to(device), threat.to(device)
    intent, action = intent.to(device), action.to(device)
    positions, velocities = positions.to(device), velocities.to(device)

    adj = build_adjacency_from_kinematics(positions, velocities).to(device)
    out = model(node_feats, adj)

    loss = (
        F.mse_loss(out['threat'], threat)
        + F.cross_entropy(out['intent_logits'], intent)
        + F.cross_entropy(out['action_logits'], action)
    )
    optimizer.zero_grad(); loss.backward(); optimizer.step()

    if step % 500 == 0:
        print(f'step {step}: loss={loss.item():.4f}')

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/hydra-net-data/models/stage3_swarm.pt')
print('Stage 3 saved.')